In [1]:
import sys
from pathlib import Path
import torch
from transformers import (
    AutoModelForSequenceClassification, 
    AutoTokenizer, 
    TrainingArguments,
    DataCollatorWithPadding
)
# Setup Root e Import
ROOT = Path.cwd().resolve().parent
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

from project_paths import get_paths
from teacher_finetune_headtail import build_teacher_tokenizer
from distillation import (
    TinyBERTPhase1, 
    Phase1DistillationTrainer, 
    patch_model_for_unnormalized_attention
)

# Paths
paths = get_paths(ROOT)
DATA_DIR = paths.data_processed

print(f"Data Source: {DATA_DIR}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data Source: C:\Users\cola0\Desktop\nlp.project-Colangelo-2526\data\processed
Device: cuda


In [2]:
from datasets import load_dataset


TEACHER_NAME = "bert-base-uncased"
tokenizer = build_teacher_tokenizer(TEACHER_NAME)
collator = DataCollatorWithPadding(tokenizer)


print("Loading from Parquet files...")
train_ds = load_dataset("parquet", data_files=str(DATA_DIR / "train.parquet"))["train"]

print(f"Train size: {len(train_ds)}")

Loading from Parquet files...
Train size: 231423


In [3]:
cols_to_keep = ["input_ids", "attention_mask", "token_type_ids", "labels"]
cols_to_remove = [col for col in train_ds.column_names if col not in cols_to_keep]

train_ds = train_ds.remove_columns(cols_to_remove)


In [4]:
TEACHER_PATH = str(paths.checkpoints / "bert_teacher_finetuned" / "checkpoint-18000") 
STUDENT_NAME = "huawei-noah/TinyBERT_General_4L_312D"

print("Loading Teacher...")
teacher = AutoModelForSequenceClassification.from_pretrained(TEACHER_PATH, attn_implementation="eager")

print("Loading Student Base...")
student_base = AutoModelForSequenceClassification.from_pretrained(STUDENT_NAME, num_labels=2, attn_implementation="eager")

print("Patching models for pre-softmax attention...")
patch_model_for_unnormalized_attention(teacher)
patch_model_for_unnormalized_attention(student_base)

teacher = teacher.to(device)

student_phase1 = TinyBERTPhase1(student_model=student_base).to(device)

Loading Teacher...
Loading Student Base...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Patching models for pre-softmax attention...
Model bert patched: now output_attentions returns pre-softmax logits.
Model bert patched: now output_attentions returns pre-softmax logits.


In [ ]:
PHASE1_OUTPUT = paths.checkpoints / "tinybert_phase1"

training_args = TrainingArguments(
    output_dir=str(PHASE1_OUTPUT),
    per_device_train_batch_size=8,      # debug/stabilità
    gradient_accumulation_steps=4, # Alza al massimo che la VRAM ti concede per stabilizzare la MSE
    num_train_epochs=8,             # Jiao et al. suggeriscono molte epoche per questa fase
    learning_rate=5e-5,             # LR costante, NIENTE LLRD qui
    fp16=False,                      # Risparmia VRAM
    #bf16=True,
    logging_steps=10,
    save_strategy="no",             # Disabilitiamo i salvataggi automatici per evitare di salvare le matrici di proiezione
    report_to="none",
    logging_nan_inf_filter=False,
    dataloader_num_workers=2,
    remove_unused_columns=False     # Cruciale! Altrimenti il Trainer scarta input_ids se non vede "labels" nel forward
)

In [7]:
trainer = Phase1DistillationTrainer(
    teacher_model=teacher,
    model=student_phase1,
    args=training_args,
    train_dataset=train_ds,
    data_collator=collator
)

print("Starting Intermediate Layer Distillation (Phase 1)...")
trainer.train()

Starting Intermediate Layer Distillation (Phase 1)...


Step,Training Loss
10,71.008600
20,60.346100
30,49.037500
40,39.426300
50,29.007900
60,22.032800
70,17.506700
80,15.016500
90,13.458200
100,12.047800


KeyboardInterrupt: 

In [ ]:
final_student_path = PHASE1_OUTPUT / "student_base_final"

print(f"Salvataggio del modello student  in: {final_student_path}")
student_phase1.student.save_pretrained(str(final_student_path))
tokenizer.save_pretrained(str(final_student_path))
print("Salvataggio completato")